In [ ]:
#| hide
from explRL.core import *
from explRL.agents import *

# explRL

> Brain-inspired explorer agents, and experiments on keeping large teams of language-model agents effective and safe

explRL began as a Python port of the navigation model in [*A neural circuit architecture for rapid learning in goal-directed navigation*](https://doi.org/10.1016/j.neuron.2024.04.036) by Chuntao Dan, Brad K. Hulse, Ramya Kappagantula, Vivek Jayaraman and Ann M. Hermundstad (*Neuron*, 2024; open access): brain-inspired agents that explore a novel environment while mapping its value landscape, combining autonomous navigation, unsupervised environment mapping and reinforcement learning coupled with internal drives. The starting code was ported and modified from Ann M. Hermundstad's MATLAB scripts, and the package reproduces the paper's flexible policy (Fig. 2d-e), its pre-structured policy (Fig. 2h) and its circuit model (Figs. 3-5).

From Phase 5 on, the repository became a research program on teams of agents: numpy teams of explorer agents sharing a foraging world (Phases 5-9), then teams of up to 64 Claude Haiku 5.5 agents, and reinforcement learning of a 3B open model, on tasks no single agent can solve (Phases 10-18). The question throughout is what keeps a large team effective and safe when some of its members are wrong or lying.

## Multi-agent research: large teams of language-model agents

The results are written up in two preprints by Chuntao Dan and Claude Opus 5.5 (Anthropic): [*Guards, Squads and Incentives: Keeping Large Teams of Language-Model Agents Effective and Safe*](https://github.com/Positron00/explRL/blob/main/paper/main.pdf) and [*Simulate First? What Cheap Rule-Agent Worlds Predict About Teams of Language-Model Agents*](https://github.com/Positron00/explRL/blob/main/paper2/main.pdf). The [highlights page](highlights.ipynb) redraws their main figures from the saved per-seed results. Five findings, strongest first:

1. **Verification works as an assigned role.** A quarter of the team assigned as guards, who post, confirm and contest clues and never solve, recovered 72% of the damage done by an equal number of liars with no ground truth (a track record computed from the true answers recovered 34%), and beat the unprotected team at every size from 8 to 64 Claude Haiku 5.5 agents.
2. **Splitting a large team makes verification scale.** Guards can contest only the clues they hold, so their advantage shrank as teams grew; squads of eight with their own guards made it grow instead, from 0.26 at 16 agents to 0.34 at 64.
3. **Larger teams are more gullible.** The share of answers built on a false clue rose from 0.79 at 8 agents to 0.91 at 32: agents treat a posted clue as settled.
4. **Copies of one model do not divide labour on their own.** No agent volunteered for an unassigned role, team-reward RL of a 3B model did not create one, and the reward decided whether trained copies kept cooperating.
5. **Move information; do not add work.** Mechanisms that pass on or certify information at no cost in turns helped; those that add work to every agent's turn or demand obedience backfired.

Every analysis plan was frozen and committed before its runs, every API reply is cached so reruns are byte-identical, and every number and figure in the preprints is regenerated from the saved results ([`paper/make_assets.py`](https://github.com/Positron00/explRL/blob/main/paper/make_assets.py), [`paper2/make_assets.py`](https://github.com/Positron00/explRL/blob/main/paper2/make_assets.py), [`tools/figures.py`](https://github.com/Positron00/explRL/blob/main/tools/figures.py)). Figures show mean ± SEM, with an asterisk at p < 0.05. The [literature review](https://github.com/Positron00/explRL/blob/main/specs/multiagent/LITERATURE.md) checks each claim against published work.

## Install

The package is not on PyPI yet. Install from a clone of the repository:

```sh
git clone https://github.com/Positron00/explRL.git
cd explRL
pip install -e .              # the package
pip install -e ".[dev]"       # notebooks, tests and docs (nbdev)
pip install -e ".[llm]"       # live Claude runs (the anthropic SDK)
pip install -e ".[train]"     # RL of the 3B model (torch, transformers, peft; vLLM on Linux)
```

## Running the experiments

**The navigation paper.** The end-to-end reproduction of Fig. 2d-e is [`nbs/demo.ipynb`](demo.ipynb), with step-by-step instructions in [`specs/demo/RUN.md`](https://github.com/Positron00/explRL/blob/main/specs/demo/RUN.md). From the repo root:

```sh
pip install -e ".[dev]"
nbdev-export
nbdev-test --n-workers 0 nbs/demo.ipynb                # quick (~2 s) → results/demo/quick/
nbdev-test --n-workers 0 --flags slow nbs/demo.ipynb   # full (~40 s) → results/demo/full/
```

You can also open the notebook in Jupyter and run all cells (include `#| slow` cells for the full 240 s × 20-run demo). Outputs are written under `results/demo/` (gitignored). If `nbdev-test` hits a multiprocessing error, keep `--n-workers 0`. The pre-structured policy (Fig. 2h) is in [`nbs/demo_prestructured.ipynb`](demo_prestructured.ipynb) ([`specs/prestructured/RUN.md`](https://github.com/Positron00/explRL/blob/main/specs/prestructured/RUN.md)) and the circuit model (Figs. 3-5) in [`nbs/demo_circuit.ipynb`](demo_circuit.ipynb) ([`specs/circuit/RUN.md`](https://github.com/Positron00/explRL/blob/main/specs/circuit/RUN.md)).

**The multi-agent phases.** Each phase has its specification, results and run guide under `specs/multiagent/` ([the plan](https://github.com/Positron00/explRL/blob/main/specs/multiagent/README.md)):

| Phase | Question | Notebooks | Run guide |
|---|---|---|---|
| 5 | A shared interface and training loop for teams of agents | [demo_multi](demo_multi.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase5/RUN.md) |
| 6 | A team environment with incentives | [demo_team](demo_team.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase6/RUN.md) |
| 7 | Harnesses: communication channels and memory | [demo_harness](demo_harness.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase7/RUN.md) |
| 8 | A benchmark: team size against channel, pre-registered | [demo_bench](demo_bench.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase8/RUN.md) |
| 9 | Safety: liars and defences, pre-registered | [demo_safety](demo_safety.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase9/RUN.md) |
| 10 | Claude agents on text foraging and a task board | [demo_llm](demo_llm.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase10/RUN.md) |
| follow-up | Faster live runs, team totals, rounds and squads, matched effort | [demo_followup](demo_followup.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/speedup/RUN.md) |
| 11 | The clue board: information as the bottleneck | [demo_clue](demo_clue.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase11/RUN.md) |
| 12 | Better instructions for Claude, and RL of a 3B open model | [demo_coop](demo_coop.ipynb), [demo_rl](demo_rl.ipynb), [colab_clue_rl](colab_clue_rl.ipynb) (Colab) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase12/RUN.md) |
| 13 | From diagnosis to fixes, with a HiddenBench replication | [demo_fix](demo_fix.ipynb), [demo_hb](demo_hb.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase13/RUN.md) |
| 14 | What makes trust information pay, and do the fixes scale? | [demo_trust](demo_trust.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase14/RUN.md) |
| 15 | Why larger Claude teams lose more to liars (replays) | [demo_liar](demo_liar.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase15/RUN.md) |
| 16 | More training seeds for the incentive result | [demo_rl16](demo_rl16.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase16/RUN.md) |
| 17 | RL for larger teams | [demo_rl17](demo_rl17.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase17/RUN.md) |
| 18 | Colony rules: guards, squads and trained roles | [demo_colony](demo_colony.ipynb), [colony32](colony32.ipynb), [colony64](colony64.ipynb), [colony4](colony4.ipynb), [demo_rl18](demo_rl18.ipynb) | [RUN.md](https://github.com/Positron00/explRL/blob/main/specs/multiagent/phase18/RUN.md) |

Cells that call Claude are flagged `llm` and skipped by default; they call the API only with `EXPLRL_LLM_LIVE=1`, reading the key from `ANTHROPIC_API_KEY` in the environment or in a git-ignored `.env.local`. GPU training runs in the Colab notebook. Results, response caches and trained adapters are kept out of git, so most phase notebooks rebuild their reports only where those exist; each run guide gives the commands, the costs and how to rerun from the cache at no cost, and the [highlights](highlights.ipynb) notebook redraws the main figures from a committed extract. The write-up notebook [`nbs/demo_multiagent.ipynb`](demo_multiagent.ipynb) writes [`results/multiagent/REPORT.md`](https://github.com/Positron00/explRL/blob/main/results/multiagent/REPORT.md).

A quick check of the multi-agent loop (Phase 5):

```sh
nbdev-test --n-workers 0 nbs/demo_multi.ipynb                # quick (~3 s) → results/multiagent/demo_multi/quick/
nbdev-test --n-workers 0 --flags slow nbs/demo_multi.ipynb   # full (~15 s) → results/multiagent/demo_multi/full/
```

## How to use

This package is in early development (alpha). **Single agents** (the paper's three variants): the flexible policy (`explRL.basis`, `flexibleAgent`, policy-gradient RL), the pre-structured setpoint policy with bump jumps (`explRL.prestructured`) and the circuit model with Hebbian compass and goal weights (`explRL.circuit`); shared pieces are `explRL.params`, `explRL.envs` (heading and zone arenas), `explRL.train` and the `explRL.core` loaders. **Teams**: `explRL.multi` and `explRL.members` run teams on one event queue; the foraging benchmark and its safety study are in the `explRL.bench_*` and `explRL.safety_*` modules; the clue board, its channels and its runners in `explRL.clueboard`, `explRL.clue_agents` and `explRL.clue_runs`; colony rules, guards and squads in `explRL.clue_colony`, `explRL.colony_agents` and `explRL.colony_board`; the Claude client and its response cache in `explRL.llm_client`; RL of the 3B model in `explRL.clue_rl` and `explRL.rl_torch`; and the publication figures in `explRL.figs_team` and `explRL.figs_rl`. The documentation site lists every module.

For the full Fig. 2d-e reproduction, see **Running the experiments** above.

Example (build an agent and arena, then train briefly):

In [ ]:
from math import pi
from explRL.core import loadAgent, loadEnv
from explRL.train import train_agent, evaluatePolicy

agent = loadAgent(goal=pi, seed=0)
env = loadEnv(goal=pi)
hist = train_agent(agent, env, Ttot=2.0)   # the paper uses 240 s
policy = evaluatePolicy(agent)            # drift rate, fixation duration and turn bias per heading


Example (rule agents on an 8-agent clue board, offline and deterministic: with no channel nobody can solve, a shared board pools the clues, and two scripted liars on that board cost the honest agents more than it gains them):

In [ ]:
from explRL.clue_runs import runClueNumpy
from explRL.llm_runs import LLMConfig

for channel, nLiar in (('none', 0), ('blackboard', 0), ('blackboard', 2)):
    cfg = LLMConfig(task='clue', channel=channel, N=8, nLiar=nLiar, turns=40, world=dict(m=2, r=1, R=30))
    r = runClueNumpy(cfg, 900)[0]
    print(f"{channel}, {nLiar} liars: honest welfare {r['welfareHonest']:.3f} per agent-turn, {r['solved']} subtasks solved")
# none, 0 liars: honest welfare 0.000 per agent-turn, 0 subtasks solved
# blackboard, 0 liars: honest welfare 0.500 per agent-turn, 73 subtasks solved
# blackboard, 2 liars: honest welfare -0.217 per agent-turn, 8 subtasks solved

In [ ]:
#| hide
# the printed values in the comments above stay true
rows = [runClueNumpy(LLMConfig(task='clue', channel=c, N=8, nLiar=n, turns=40, world=dict(m=2, r=1, R=30)), 900)[0]
        for c, n in (('none', 0), ('blackboard', 0), ('blackboard', 2))]
assert [round(r['welfareHonest'], 3) for r in rows] == [0.0, 0.5, -0.217] and [r['solved'] for r in rows] == [0, 73, 8]